# 延后初始化

在定义网络的时侯：

* 没有指定输入的维度
* 添加层的时候没有指定前一维的输出
* 初始化参数的时候，没有足够的信息来确定模型应该有多少参数

实例化一个多层感知机，输入的维数是未知的，网络不知道输入层权重的维数，数据通过网络的时候才能最终初始化参数

In [2]:
import torch
from torch import nn

# 1. 定义网络：使用 LazyLinear 实现延后初始化
# 注意：在这里我们只指定了输出维度（256和10），完全没有告诉网络输入维度是多少
net = nn.Sequential(
    nn.LazyLinear(256),  # 隐藏层：不指定输入维度，只指定输出256
    nn.ReLU(),
    nn.LazyLinear(10)    # 输出层：不指定输入维度，只指定输出10
)  # 只指定out_features

print("=== 传入数据前（延后初始化阶段） ===")
print(net)
# 此时打印网络，你会看到输出中带有 <uninitialized> 
# 因为框架目前还没有见过任何数据，不知道输入特征的形状。

# 2. 准备数据
# 假设我们有一个批量大小为 2，特征维度为 20 的输入数据
X = torch.rand(2, 20)

# 3. 第一次前向传播：触发真正的初始化
print("\n=== 将数据传入网络进行前向传播 ===")
out = net(X)

print("\n=== 传入数据后（参数已完成初始化） ===")
print(net)
# 再次打印网络，你会发现 <uninitialized> 消失了。
# 框架根据输入 X 的形状（特征维度为20），自动推断出：
# 第一个 LazyLinear 需要接收 20 维的输入。
# 第二个 LazyLinear 需要接收 256 维的输入（因为上一层的输出是 256）。

=== 传入数据前（延后初始化阶段） ===
Sequential(
  (0): LazyLinear(in_features=0, out_features=256, bias=True)
  (1): ReLU()
  (2): LazyLinear(in_features=0, out_features=10, bias=True)
)

=== 将数据传入网络进行前向传播 ===

=== 传入数据后（参数已完成初始化） ===
Sequential(
  (0): Linear(in_features=20, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=10, bias=True)
)
